# TF-coil tables and global quantities

Reads the **saved outputs** of the four `postprocessing.ipynb` notebooks,
displays the three TF-component tables and the DPA/TBR table, and generates
`models/comparison/tf_coil_tables.tex`.

1. Run the postprocessing for the configurations you want to update and **save** the notebooks with their outputs.
2. Open this notebook and select **Run All**.
3. Include the generated `.tex` file in your LaTeX document.

Values and relative uncertainties are copied as printed in the source notebooks;
they are not recalculated from the already rounded standard deviations.
This notebook uses the Python standard library and does not require `.h5` files
or OpenMC. To use new simulation results, first rerun the postprocessing and
save its outputs.

The LaTeX document preamble requires `amsmath`, `booktabs`, and `makecell`.

## Configurations and paths

The order below determines the table columns. Paths are detected when this
notebook is started from the project directory, `models`, `models/comparison`,
or the directory containing the project.
You can also set `PROJECT_ROOT` explicitly.

In [9]:
import json
import math
import re
from dataclasses import dataclass
from html import escape
from pathlib import Path

PROJECT_ROOT = None  # Example: Path("/home/davide/PoliTo/fusion_fuel_concepts_explorer")


def find_project_root(start):
    for parent in (start, *start.parents):
        for candidate in (parent, parent / "fusion_fuel_concepts_explorer"):
            if (candidate / "models/d_t/postprocessing.ipynb").is_file():
                return candidate.resolve()
    raise FileNotFoundError(
        "Project not found: set PROJECT_ROOT to the repository path."
    )


PROJECT_ROOT = (
    Path(PROJECT_ROOT).expanduser().resolve()
    if PROJECT_ROOT is not None
    else find_project_root(Path.cwd())
)
OUTPUT_PATH = PROJECT_ROOT / "models/comparison/tf_coil_tables.tex"

CONFIGURATIONS = [
    ("D–T (FLiBe)", "d_t", r"\makecell{D--T\\(FLiBe)}"),
    ("D–³He (FLiBe)", "d_he3", r"\makecell{D--$^3$He\\(FLiBe)}"),
    (
        "D–³He (Be–water–Inconel)",
        "d_he3/inconel_water_config",
        r"\makecell{D--$^3$He\\(Be--water--\\Inconel)}",
    ),
    (
        "D–³He (water–Inconel)",
        "d_he3/no_be",
        r"\makecell{D--$^3$He\\(water--\\Inconel)}",
    ),
]
COMPONENTS = [
    ("tf_coil_magnet", "TF-coil winding pack", "windingpack"),
    ("tf_coil_insulation", "TF-coil fiberglass insulation", "insulation"),
    ("tf_coil_case", "TF-coil magnet case", "magnetcase"),
]
# Key, display label, LaTeX label, display unit, LaTeX unit.
FLUX_ROWS = [
    ("total_flux", "Total neutron flux", "Total neutron flux", "n cm⁻² s⁻¹", r"$\mathrm{n\,cm^{-2}\,s^{-1}}$"),
    ("above_7", "Flux, Eₙ > 7 MeV", r"Flux, $E_n>7\,\mathrm{MeV}$", "n cm⁻² s⁻¹", r"$\mathrm{n\,cm^{-2}\,s^{-1}}$"),
    ("1_to_7", "Flux, 1 ≤ Eₙ ≤ 7 MeV", r"Flux, $1\leq E_n\leq7\,\mathrm{MeV}$", "n cm⁻² s⁻¹", r"$\mathrm{n\,cm^{-2}\,s^{-1}}$"),
    ("below_1", "Flux, Eₙ < 1 MeV", r"Flux, $E_n<1\,\mathrm{MeV}$", "n cm⁻² s⁻¹", r"$\mathrm{n\,cm^{-2}\,s^{-1}}$"),
]
COMPONENT_ROWS = FLUX_ROWS + [
    ("power", "Deposited power", "Deposited power", "MW", r"$\mathrm{MW}$"),
]
GLOBAL_ROWS = [
    ("dpa", "DPA on YBCO", "DPA on YBCO", "dpa FPY⁻¹", r"$\mathrm{dpa\,FPY^{-1}}$"),
    ("tbr", "Total TBR", "Total TBR", "--", "--"),
]
print(f"Project: {PROJECT_ROOT}")
print(f"LaTeX file: {OUTPUT_PATH}")

Project: /home/davide/PoliTo/fusion_fuel_concepts_explorer
LaTeX file: /home/davide/PoliTo/fusion_fuel_concepts_explorer/models/comparison/tf_coil_tables.tex


## Read and validate saved outputs

Cells are identified by component names and output labels, independently of
their position in the source notebook. Each configuration must provide
17 measurements: four fluxes and one deposited power per component,
plus annual DPA and total TBR. Missing or duplicate outputs stop execution.

In [10]:
NUMBER = r"[+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?"
COMPONENT_KEYS = {item[0] for item in COMPONENTS}
FLUX_OUTPUT_LABELS = {
    "total_flux": r"Total flux",
    "above_7": r"Flux\s*>\s*7 MeV",
    "1_to_7": r"Flux\s+1[–−-]7 MeV",
    "below_1": r"Flux\s*<\s*1 MeV",
}


@dataclass(frozen=True)
class Measurement:
    mean: str
    std_dev: str
    relative_percent: str
    cell_number: int


def text_content(value):
    return value if isinstance(value, str) else "".join(value)


def saved_output_text(cell):
    parts = []
    for output in cell.get("outputs", []):
        if output.get("output_type") == "stream":
            parts.append(text_content(output.get("text", "")))
        elif output.get("output_type") in {"execute_result", "display_data"}:
            parts.append(text_content(output.get("data", {}).get("text/plain", "")))
    return "\n".join(parts)


def read_results(path):
    if not path.is_file():
        raise FileNotFoundError(f"Source notebook not found: {path}")
    notebook = json.loads(path.read_text(encoding="utf-8"))
    records = {}

    def add(component, metric, match, cell_number):
        key = (component, metric)
        if key in records:
            raise ValueError(f"{path}, cell {cell_number}: duplicate output for {key}.")
        mean, std_dev, relative = match
        if any(not math.isfinite(float(v)) or float(v) < 0 for v in match):
            raise ValueError(f"{path}, cell {cell_number}: invalid values for {key}.")
        records[key] = Measurement(mean, std_dev, relative, cell_number)

    def exactly_one(pattern, text, description, cell_number):
        matches = re.findall(pattern, text, flags=re.MULTILINE)
        if len(matches) != 1:
            raise ValueError(
                f"{path}, cell {cell_number}: expected one output for {description}, "
                f"found {len(matches)}. Run and save the postprocessing notebook."
            )
        return matches[0]

    for cell_number, cell in enumerate(notebook.get("cells", []), 1):
        if cell.get("cell_type") != "code":
            continue
        source = text_content(cell.get("source", ""))
        text = saved_output_text(cell)
        for match in re.findall(
            rf"The reactor has a TBR of\s+({NUMBER})\s*±\s*({NUMBER})\s*\(({NUMBER})\s*%\)", text
        ):
            add(None, "tbr", match, cell_number)

        for component, mean, std_dev, relative in re.findall(
            rf"Results for tally ['\"]q_vol_(tf_coil_\w+)['\"]:\s*"
            rf"Power deposited:\s*({NUMBER})\s*±\s*({NUMBER})\s*MW\s*\(({NUMBER})\s*%\)", text
        ):
            if component in COMPONENT_KEYS:
                add(component, "power", (mean, std_dev, relative), cell_number)

        components = re.findall(
            r"\bname\s*=\s*['\"]neutron_spectra_(tf_coil_\w+)['\"]", source
        )
        components = [component for component in components if component in COMPONENT_KEYS]
        if components and re.search(r"^Total flux\s*=", text, re.MULTILINE):
            if len(components) != 1:
                raise ValueError(
                    f"{path}, cell {cell_number}: put plot_flux_spectrum calls "
                    "in separate cells to associate each output with its component."
                )
            for metric, label in FLUX_OUTPUT_LABELS.items():
                match = exactly_one(
                    rf"^{label}\s*=\s*({NUMBER})\s*±\s*({NUMBER})\s*n/cm²/s\s*\(({NUMBER})\s*%\)",
                    text, metric, cell_number,
                )
                add(components[0], metric, match, cell_number)

        if re.search(r"^DPA\s*=", text, re.MULTILINE):
            mean, std_dev = exactly_one(
                rf"^DPA\s*=\s*({NUMBER})\s*±\s*({NUMBER})\s*$", text, "annual DPA", cell_number
            )
            # Match the original Italian label printed by the source notebooks.
            relative = exactly_one(
                rf"^Errore relativo\s*=\s*({NUMBER})\s*%\s*\(1σ\)", text, "DPA uncertainty", cell_number
            )
            add(None, "dpa", (mean, std_dev, relative), cell_number)

    expected = {
        (component, row[0]) for component in COMPONENT_KEYS for row in COMPONENT_ROWS
    } | {(None, row[0]) for row in GLOBAL_ROWS}
    missing = expected - records.keys()
    if missing:
        labels = ", ".join(f"{component or 'global'}/{metric}" for component, metric in sorted(missing, key=str))
        raise ValueError(
            f"{path}: missing outputs: {labels}. Run and save the source notebook with its outputs."
        )
    return records


results = {}
for name, config, _ in CONFIGURATIONS:
    path = PROJECT_ROOT / "models" / config / "postprocessing.ipynb"
    results[config] = read_results(path)
    print(f"{name}: {len(results[config])} measurements read from {path.relative_to(PROJECT_ROOT)}")
print(f"Total: {sum(map(len, results.values()))} measurements with relative uncertainties at 1σ.")

D–T (FLiBe): 17 measurements read from models/d_t/postprocessing.ipynb
D–³He (FLiBe): 17 measurements read from models/d_he3/postprocessing.ipynb
D–³He (Be–water–Inconel): 17 measurements read from models/d_he3/inconel_water_config/postprocessing.ipynb
D–³He (water–Inconel): 17 measurements read from models/d_he3/no_be/postprocessing.ipynb
Total: 68 measurements with relative uncertainties at 1σ.


In [11]:
def latex_number(value):
    if "e" in value.lower():
        mantissa, exponent = value.lower().split("e")
        return rf"${mantissa}\cdot 10^{{{int(exponent)}}}$"
    return value


def html_number(value):
    if "e" in value.lower():
        mantissa, exponent = value.lower().split("e")
        return f"{escape(mantissa)} × 10<sup>{int(exponent)}</sup>"
    return escape(value)


class ResultTable:
    def __init__(self, title, component, rows):
        self.title, self.component, self.rows = title, component, rows

    def _repr_html_(self):
        headers = ["Quantity", "Unit"] + [name for name, _, _ in CONFIGURATIONS]
        head = "".join(f"<th>{escape(label)}</th>" for label in headers)
        body = []
        for metric, label, _, unit, _ in self.rows:
            values = []
            for _, config, _ in CONFIGURATIONS:
                measurement = results[config][(self.component, metric)]
                values.append(
                    f"<td style='white-space:nowrap'>{html_number(measurement.mean)} "
                    f"({escape(measurement.relative_percent)}%)</td>"
                )
            body.append(
                f"<tr><th style='text-align:left'>{escape(label)}</th>"
                f"<td style='white-space:nowrap'>{escape(unit)}</td>{''.join(values)}</tr>"
            )
        return (
            f"<div style='overflow-x:auto'><h3>{escape(self.title)}</h3>"
            f"<table><thead><tr>{head}</tr></thead><tbody>{''.join(body)}</tbody></table>"
            "<p>Parentheses: relative uncertainty at 1σ (%).</p></div>"
        )

    def __repr__(self):
        lines = [self.title, " | ".join(["Quantity", "Unit"] + [c[0] for c in CONFIGURATIONS])]
        for metric, label, _, unit, _ in self.rows:
            cells = [label, unit]
            for _, config, _ in CONFIGURATIONS:
                measurement = results[config][(self.component, metric)]
                cells.append(f"{measurement.mean} ({measurement.relative_percent}%)")
            lines.append(" | ".join(cells))
        return "\n".join(lines)

## TF-coil winding pack (`tf_coil_magnet`)

In [12]:
ResultTable("TF-coil winding pack", "tf_coil_magnet", COMPONENT_ROWS)

Quantity,Unit,D–T (FLiBe),D–³He (FLiBe),D–³He (Be–water–Inconel),D–³He (water–Inconel)
Total neutron flux,n cm⁻² s⁻¹,2.747 × 106 (6.12%),6.642 × 104 (7.05%),1.112 × 104 (1.78%),1.200 × 104 (1.77%)
"Flux, Eₙ > 7 MeV",n cm⁻² s⁻¹,7.632 × 103 (4.81%),8.763 × 101 (9.31%),9.138 × 101 (10.95%),9.434 × 101 (13.94%)
"Flux, 1 ≤ Eₙ ≤ 7 MeV",n cm⁻² s⁻¹,5.812 × 104 (3.86%),6.689 × 102 (2.27%),5.758 × 102 (6.19%),6.420 × 102 (6.54%)
"Flux, Eₙ < 1 MeV",n cm⁻² s⁻¹,2.681 × 106 (6.27%),6.566 × 104 (7.14%),1.045 × 104 (1.86%),1.126 × 104 (1.84%)
Deposited power,MW,0.006 (6.18%),0.0003 (24.07%),0.0004 (51.40%),0.0002 (30.03%)


## TF-coil fiberglass insulation (`tf_coil_insulation`)

In [13]:
ResultTable("TF-coil fiberglass insulation", "tf_coil_insulation", COMPONENT_ROWS)

Quantity,Unit,D–T (FLiBe),D–³He (FLiBe),D–³He (Be–water–Inconel),D–³He (water–Inconel)
Total neutron flux,n cm⁻² s⁻¹,7.464 × 106 (10.47%),1.128 × 105 (14.50%),1.322 × 104 (2.67%),1.355 × 104 (2.78%)
"Flux, Eₙ > 7 MeV",n cm⁻² s⁻¹,8.067 × 104 (54.36%),2.922 × 102 (30.82%),5.971 × 102 (54.75%),6.082 × 102 (53.75%)
"Flux, 1 ≤ Eₙ ≤ 7 MeV",n cm⁻² s⁻¹,1.177 × 105 (14.19%),1.351 × 103 (4.97%),8.187 × 102 (3.60%),7.969 × 102 (2.50%)
"Flux, Eₙ < 1 MeV",n cm⁻² s⁻¹,7.265 × 106 (10.74%),1.111 × 105 (14.71%),1.181 × 104 (1.11%),1.215 × 104 (1.53%)
Deposited power,MW,0.066 (20.95%),0.0027 (20.70%),0.0052 (58.47%),0.0061 (54.29%)


## TF-coil magnet case (`tf_coil_case`)

In [14]:
ResultTable("TF-coil magnet case", "tf_coil_case", COMPONENT_ROWS)

Quantity,Unit,D–T (FLiBe),D–³He (FLiBe),D–³He (Be–water–Inconel),D–³He (water–Inconel)
Total neutron flux,n cm⁻² s⁻¹,9.674 × 106 (5.40%),4.731 × 105 (6.82%),1.865 × 104 (13.18%),1.879 × 104 (13.09%)
"Flux, Eₙ > 7 MeV",n cm⁻² s⁻¹,2.439 × 105 (21.22%),1.502 × 103 (22.59%),2.143 × 103 (41.42%),2.155 × 103 (41.18%)
"Flux, 1 ≤ Eₙ ≤ 7 MeV",n cm⁻² s⁻¹,1.034 × 106 (20.82%),6.254 × 103 (28.27%),6.568 × 103 (34.78%),6.609 × 103 (34.61%)
"Flux, Eₙ < 1 MeV",n cm⁻² s⁻¹,8.397 × 106 (5.63%),4.653 × 105 (6.92%),9.937 × 103 (1.79%),1.003 × 104 (1.79%)
Deposited power,MW,0.600 (14.70%),0.0319 (20.11%),0.0771 (11.53%),0.0718 (11.79%)


## DPA on YBCO and total TBR

In [15]:
ResultTable("DPA on YBCO and total TBR", None, GLOBAL_ROWS)

Quantity,Unit,D–T (FLiBe),D–³He (FLiBe),D–³He (Be–water–Inconel),D–³He (water–Inconel)
DPA on YBCO,dpa FPY⁻¹,1.4834 × 10-1 (5.59%),2.4941 × 10-3 (15.94%),8.9096 × 10-4 (3.99%),9.7746 × 10-4 (5.58%)
Total TBR,--,1.0865 (0.81%),0.7767 (2.27%),0.0165 (3.48%),0.0154 (3.77%)


## LaTeX export

Running this cell creates or updates `tf_coil_tables.tex` in the same directory
as this notebook. The file contains all four tables, with the captions, units,
labels, and column order of the original text.
Both the displayed tables and the export use the same `results` dictionary.

In [16]:
UNCERTAINTY_CAPTION = (
    "Values in parentheses are one-standard-deviation relative uncertainties,\n"
    r"$100\sigma/\bar{x}$, expressed in percent."
)


def latex_table(title, label, component, rows):
    if component is None:
        caption = "DPA and TBR for the considered configurations. "
    else:
        caption = (
            r"Main neutron fluxes and deposited power in the \textbf{"
            + title + "} for the considered configurations. "
        )
    lines = [
        "%=================================================================",
        f"% {title.upper()}",
        "%=================================================================",
        "",
        r"\begin{table*}[!h]",
        r"\centering",
        r"\caption{" + caption + UNCERTAINTY_CAPTION + "}",
        r"\label{tab:" + label + "}",
        "",
        r"\begin{tabular}{@{}ll" + "c" * len(CONFIGURATIONS) + r"@{}}",
        r"\toprule",
        "Quantity",
        "& Unit",
        *["& " + heading for _, _, heading in CONFIGURATIONS],
        r"\\",
        r"\midrule",
        "",
    ]
    for index, (metric, _, row_label, _, unit) in enumerate(rows):
        if metric == "power" or (component is None and index > 0):
            lines.append(r"\midrule")
        lines.extend([row_label, "& " + unit])
        for _, config, _ in CONFIGURATIONS:
            measurement = results[config][(component, metric)]
            lines.append(
                f"& {latex_number(measurement.mean)} ({measurement.relative_percent}\\%)"
            )
        lines.extend([r"\\", ""])
    lines.extend([r"\bottomrule", r"\end{tabular}", r"\end{table*}"])
    return "\n".join(lines)


header = [
    "% Generated by models/comparison/tf_coil_tables.ipynb.",
    "% Values copied from saved outputs of the following notebooks:",
    *[f"% models/{config}/postprocessing.ipynb" for _, config, _ in CONFIGURATIONS],
    "% TF winding pack = tf_coil_magnet; insulation = tf_coil_insulation; case = tf_coil_case.",
    "% Parentheses contain the printed 1-sigma relative uncertainties (percent).",
    r"% Required LaTeX packages: amsmath, booktabs, makecell.",
]
tables = [
    latex_table(title, label, component, COMPONENT_ROWS)
    for component, title, label in COMPONENTS
]
tables.append(latex_table("Global scenario quantities", "DPA_TBR", None, GLOBAL_ROWS))
latex_text = "\n".join(header) + "\n\n" + "\n\n".join(tables) + "\n"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.write_text(latex_text, encoding="utf-8")
print(f"Saved {len(tables)} tables to: {OUTPUT_PATH}")
print("To view or copy the LaTeX source: print(latex_text)")

Saved 4 tables to: /home/davide/PoliTo/fusion_fuel_concepts_explorer/models/comparison/tf_coil_tables.tex
To view or copy the LaTeX source: print(latex_text)
